# CDI Stability Prediction: Final Machine Learning Pipeline

In [4]:
!pip install -q pytorch-tabnet catboost xgboost lightgbm imbalanced-learn scikit-learn pandas numpy

In [5]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

SEED = 42
BASELINE_THRESHOLD = 0.60
HELDOUT_SEEDS = (101, 202, 303, 404, 505)

## 1. Baseline Model (Exact Paper Replication)

**IMPORTANT NOTES ON REPLICATION**

1. **Data leakage.** The source paper fits the `MinMaxScaler` on the *entire*
   dataset before `train_test_split`. We replicate that here so the numbers match
   the publication, and correct it for all subsequent models.
2. **Threshold.** The source paper evaluates accuracy, F1 and the confusion matrix
   at a threshold of `0.60` rather than the default `0.50`. This is not stated in
   the published text; it was recovered from the authors' released implementation
   and is required to reproduce their confusion matrices. All AUC values reported
   in the manuscript are threshold-independent and unaffected by it.
3. **CatBoost seed.** The source paper does not set `random_state` for CatBoost,
   so it takes the library default of `0`. We preserve their setting in order to
   reproduce their published output exactly.
4. **Cross-validation.** The source paper reports a **single** stratified 80/20
   split and does not cross-validate. Repeated stratified cross-validation over
   held-out seeds (`101, 202, 303, 404, 505`) is introduced by **this work**, and
   is the basis of the reliability analysis in Section 3.4 of the manuscript.


In [6]:
df = pd.read_csv('cdi_dataset.csv')
if 'ID_IG' in df.columns:
    df = df.rename(columns={'ID_IG': 'ID/IG'})
df['Class'] = (df['CN'] > 40).astype(int)

X_orig = df[['EC', 'AV', 'FR', 'SSA', 'Vpore', 'ID/IG', 'N', 'O']]
y = df['Class']

# INTENTIONAL DATA LEAKAGE TO MATCH PAPER
scaler_leaky = MinMaxScaler().fit(X_orig)
X_scaled_leaky = scaler_leaky.transform(X_orig)

X_train_orig, X_test_orig, y_train, y_test = train_test_split(X_scaled_leaky, y, test_size=0.2, stratify=y, random_state=SEED)

smote = SMOTE(random_state=SEED)
X_train_orig_res, y_train_res = smote.fit_resample(X_train_orig, y_train)

baseline_models = {
    'CatBoost': CatBoostClassifier(depth=5, iterations=200, l2_leaf_reg=3, learning_rate=0.1, verbose=0),
    'LightGBM': LGBMClassifier(colsample_bytree=1.0, learning_rate=0.05, max_depth=3,
                               n_estimators=100, num_leaves=15, reg_alpha=0.1,
                               reg_lambda=1, subsample=0.8, verbose=-1, random_state=SEED),
    'GBC':      GradientBoostingClassifier(learning_rate=0.5, n_estimators=50, max_depth=7,
                                           subsample=0.8, min_samples_split=2, random_state=SEED),
}

print("=== BASELINE: SINGLE SPLIT (Seed 42) ===")
for name, model in baseline_models.items():
    model.fit(X_train_orig_res, y_train_res)
    p = model.predict_proba(X_test_orig)[:, 1]
    pred = (p >= BASELINE_THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    auc = roc_auc_score(y_test, p)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    print(f"{name:10s} | AUC: {auc:.3f} | Acc: {acc:.3f} | F1: {f1:.3f} | CM: {cm.tolist()}")

print("\n=== BASELINE: CROSS-VALIDATION ===")
for name, model in baseline_models.items():
    per_seed_scores = []
    for s in HELDOUT_SEEDS:
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=s)
        pipe = ImbPipeline([('scaler', MinMaxScaler()), ('smote', SMOTE(random_state=SEED)), ('model', model)])
        cv_auc = cross_val_score(pipe, X_orig, y, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
        per_seed_scores.append(cv_auc)
    print(f"{name:10s} | Mean CV AUC: {np.mean(per_seed_scores):.3f} ± {np.std(per_seed_scores):.3f}")

=== BASELINE: SINGLE SPLIT (Seed 42) ===
CatBoost   | AUC: 0.907 | Acc: 0.889 | F1: 0.824 | CM: [[17, 3], [0, 7]]
LightGBM   | AUC: 0.879 | Acc: 0.815 | F1: 0.615 | CM: [[18, 2], [3, 4]]
GBC        | AUC: 0.850 | Acc: 0.741 | F1: 0.588 | CM: [[15, 5], [2, 5]]

=== BASELINE: CROSS-VALIDATION ===
CatBoost   | Mean CV AUC: 0.666 ± 0.019
LightGBM   | Mean CV AUC: 0.598 ± 0.008
GBC        | Mean CV AUC: 0.651 ± 0.014


## 2. Feature Engineering & Strict No-Leakage Pipeline
For all of our improvements, we strictly enforce a **no data leakage** policy: `MinMaxScaler` is fitted exclusively on the training sets. We select the most important features (`EC`, `FR`, `SSA`, `N`, `O`) and create interaction terms (`EC/FR` and `Vpore/SSA`).

In [7]:
X_fe = df[['EC', 'FR', 'SSA', 'N', 'O']].copy()
X_fe['EC_FR'] = df['EC'] / (df['FR'] + 1e-6)
X_fe['Vpore_SSA'] = df['Vpore'] / (df['SSA'] + 1e-6)

# NO DATA LEAKAGE: We split FIRST, then scale
X_train_fe, X_test_fe, y_train, y_test = train_test_split(X_fe, y, test_size=0.2, stratify=y, random_state=SEED)
scaler_fe = MinMaxScaler()
X_train_fe_s = scaler_fe.fit_transform(X_train_fe)
X_test_fe_s = scaler_fe.transform(X_test_fe)
X_train_fe_res, y_train_res_fe = smote.fit_resample(X_train_fe_s, y_train)

## 3. Advanced Tree-Based Models
Implementing CatBoost, tuned XGBoost, and HistBoost on the engineered dataset. Note that XGBoost is evaluated using its heavily optimized threshold of `0.75`.

In [8]:
# Hyperparameters follow Table S2 of the source publication, exactly as used for
# the baseline models above, so that the only change between the baseline and the
# improved configurations is the descriptor set. HistGradientBoosting is not among
# the source study's eight models; its settings are stated in the manuscript and
# were not tuned against test-set performance.
fe_models = {
    'CatBoost':  CatBoostClassifier(depth=5, iterations=200, l2_leaf_reg=3,
                                    learning_rate=0.1, verbose=0),
    'XGBoost':   XGBClassifier(colsample_bytree=1.0, learning_rate=0.1, max_depth=3,
                               n_estimators=200, reg_alpha=0.1, reg_lambda=1,
                               subsample=1.0, eval_metric='logloss',
                               random_state=SEED),
    'HistBoost': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1,
                                                max_iter=200, l2_regularization=1.0,
                                                random_state=SEED),
}

print("=== IMPROVED MODELS: SINGLE SPLIT (Seed 42) ===")
for name, model in fe_models.items():
    model.fit(X_train_fe_res, y_train_res_fe)

    # Threshold calibrated on the original imbalanced TRAINING data.
    # (Calibrating on test data would leak; the manuscript reports AUC only,
    #  which is threshold-independent in any case.)
    p_train_orig = model.predict_proba(X_train_fe_s)[:, 1]
    best_f1_tr, best_thresh = 0, 0.50
    for t in np.arange(0.1, 0.95, 0.01):
        f1_t = f1_score(y_train, (p_train_orig >= t).astype(int))
        if f1_t > best_f1_tr:
            best_f1_tr, best_thresh = f1_t, t

    p_test = model.predict_proba(X_test_fe_s)[:, 1]
    pred   = (p_test >= best_thresh).astype(int)
    cm     = confusion_matrix(y_test, pred, labels=[0, 1])
    print(f"{name:10s} | Threshold: {best_thresh:.2f} | "
          f"AUC: {roc_auc_score(y_test, p_test):.3f} | "
          f"Acc: {accuracy_score(y_test, pred):.3f} | "
          f"F1: {f1_score(y_test, pred):.3f} | CM: {cm.tolist()}")

print("\n=== IMPROVED MODELS: CROSS-VALIDATION ===")
for name, model in fe_models.items():
    per_seed_scores = []
    for s in HELDOUT_SEEDS:
        cv   = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=s)
        pipe = ImbPipeline([('scaler', MinMaxScaler()),
                            ('smote',  SMOTE(random_state=SEED)),
                            ('model',  model)])
        per_seed_scores.append(
            cross_val_score(pipe, X_fe, y, cv=cv, scoring='roc_auc', n_jobs=-1).mean())
    print(f"{name:10s} | Mean CV AUC: {np.mean(per_seed_scores):.3f} "
          f"± {np.std(per_seed_scores):.3f}")

print("\nExpected (manuscript Table 3):")
print("  single split  CatBoost 0.893 | XGBoost 0.921 | HistBoost 0.921")
print("  cross-val     CatBoost 0.726 | XGBoost 0.696 | HistBoost 0.711")

=== IMPROVED MODELS: SINGLE SPLIT (Seed 42) ===
CatBoost   | Threshold: 0.10 | AUC: 0.893 | Acc: 0.778 | F1: 0.700 | CM: [[14, 6], [0, 7]]
XGBoost    | Threshold: 0.28 | AUC: 0.921 | Acc: 0.815 | F1: 0.706 | CM: [[16, 4], [1, 6]]
HistBoost  | Threshold: 0.42 | AUC: 0.921 | Acc: 0.815 | F1: 0.706 | CM: [[16, 4], [1, 6]]

=== IMPROVED MODELS: CROSS-VALIDATION ===
CatBoost   | Mean CV AUC: 0.726 ± 0.010
XGBoost    | Mean CV AUC: 0.696 ± 0.014
HistBoost  | Mean CV AUC: 0.711 ± 0.012

Expected (manuscript Table 3):
  single split  CatBoost 0.893 | XGBoost 0.921 | HistBoost 0.921
  cross-val     CatBoost 0.726 | XGBoost 0.696 | HistBoost 0.711


## 4. Modern Deep Learning: PyTorch TabNet

TabNet applies sequential attention at each decision step to select a sparse
subset of descriptors, combining the capacity of a neural network with
instance-wise feature selection. To our knowledge this is its first application
to CDI desalination stability.

We report **AUC only**. AUC is threshold-independent, so no decision-threshold
choice enters the reported figure.

In [9]:
# =====================================================================
# TabNet - single split (manuscript Table 3, row 7)
# =====================================================================
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

def make_tabnet(seed=42):
    return TabNetClassifier(
        n_d=4, n_a=4, n_steps=3,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        scheduler_params={"step_size": 50, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', seed=seed, verbose=0, device_name='auto')


def tabnet_fit_auc(seed, monitoring):
    """Fit TabNet and return test AUC.

    monitoring=True additionally passes eval_set on the TRAINING partition with
    patience == max_epochs, so no early stopping occurs. It computes a diagnostic
    metric and nothing else - no gradient, stopping rule or hyperparameter depends
    on it - but it consumes pseudo-random draws and therefore shifts the network
    initialisation. See Section 3.3 of the manuscript.
    """
    model = make_tabnet(seed=seed)
    kw = dict(X_train=X_train_fe_res, y_train=y_train_res_fe.values,
              max_epochs=100, batch_size=32, virtual_batch_size=16,
              num_workers=0, drop_last=False)
    if monitoring:
        kw.update(eval_set=[(X_train_fe_res, y_train_res_fe.values)],
                  eval_metric=['auc'], patience=100)
    else:
        kw.update(patience=0)
    model.fit(**kw)
    return roc_auc_score(y_test, model.predict_proba(X_test_fe_s)[:, 1])


auc_A = tabnet_fit_auc(42, monitoring=False)
auc_B = tabnet_fit_auc(42, monitoring=True)

print("=== TabNet, single split, seed 42 ===")
print(f"  Configuration A (no monitoring argument):   AUC {auc_A:.3f}")
print(f"  Configuration B (with monitoring argument): AUC {auc_B:.3f}")
print(f"  Difference: {abs(auc_B - auc_A):.3f}")
print("\nExpected (manuscript Table 3): 0.893 / 0.950")

Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_auc = 0.92969
=== TabNet, single split, seed 42 ===
  Configuration A (no monitoring argument):   AUC 0.893
  Configuration B (with monitoring argument): AUC 0.950
  Difference: 0.057

Expected (manuscript Table 3): 0.893 / 0.950


## 5. Initialisation study — two fitting configurations (manuscript Table 2, Fig. 3a)

The single split is held completely fixed. Only the network initialisation seed
varies, under each of the two fitting configurations of Section 4.

If the 0.057 gap at seed 42 reflected something real about the two configurations,
their distributions over ten seeds would differ. They do not — which is the
manuscript's central point about single-split estimates at this sample size.

⚠️ Twenty TabNet fits. A few minutes on a GPU runtime.

In [10]:
INIT_SEEDS = [0, 1, 7, 13, 42, 101, 202, 303, 404, 505]
results = {}

for label, monitoring in [('A: without monitoring argument', False),
                          ('B: with monitoring argument',    True)]:
    aucs = np.array([tabnet_fit_auc(s, monitoring) for s in INIT_SEEDS])
    results[label] = aucs
    print(label)
    print('   ' + '  '.join(f'{s}:{a:.3f}' for s, a in zip(INIT_SEEDS, aucs)))
    print(f'   mean {aucs.mean():.3f} ± {aucs.std(ddof=1):.3f}   '
          f'range {aucs.min():.3f}–{aucs.max():.3f}\n')

from scipy import stats
A, B = results['A: without monitoring argument'], results['B: with monitoring argument']
t_stat, p_welch = stats.ttest_ind(A, B, equal_var=False)
_, p_mw = stats.mannwhitneyu(A, B)
pooled = np.concatenate([A, B])

print(f'Welch t = {t_stat:.2f}, p = {p_welch:.2f}   Mann–Whitney p = {p_mw:.2f}')
print(f'Pooled over 20 runs: {pooled.mean():.3f} ± {pooled.std(ddof=1):.3f}   '
      f'range {pooled.min():.3f}–{pooled.max():.3f}')
print(f'\nThe two configurations are not statistically distinguishable, yet seed 42')
print(f'differs by {abs(B[4] - A[4]):.3f} between them.')
print('Configuration B has the LOWER mean despite containing the HIGHER maximum.')

A: without monitoring argument
   0:0.814  1:0.864  7:0.914  13:0.864  42:0.893  101:0.900  202:0.929  303:0.893  404:0.914  505:0.900
   mean 0.889 ± 0.033   range 0.814–0.929

Stop training because you reached max_epochs = 100 with best_epoch = 98 and best_val_0_auc = 0.97719
Stop training because you reached max_epochs = 100 with best_epoch = 93 and best_val_0_auc = 0.96516
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_auc = 0.92188
Stop training because you reached max_epochs = 100 with best_epoch = 94 and best_val_0_auc = 0.95125
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_auc = 0.92969
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_auc = 0.93156
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_auc = 0.98266
Stop training because you reached max_epochs = 100 with best_epoch = 98 and best_val_0_auc = 0.97609
Stop training 

## 6. TabNet cross-validation (manuscript Table 3, row 7)

Scaler and SMOTE refitted inside every fold. No `eval_set`, since passing the
validation fold there would leak it into training.

⚠️ 150 model fits, roughly 30 minutes on a GPU runtime.

In [ ]:
import time

X_arr, y_arr = X_fe.values, y.values
tabnet_cv, t0 = [], time.time()

for s in HELDOUT_SEEDS:
    cv, fold_aucs = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=s), []
    for tr, va in cv.split(X_arr, y_arr):
        X_tr, X_va = X_arr[tr], X_arr[va]
        y_tr, y_va = y_arr[tr], y_arr[va]

        scaler = MinMaxScaler().fit(X_tr)                    # training fold only
        X_tr, X_va = scaler.transform(X_tr), scaler.transform(X_va)
        X_tr, y_tr = SMOTE(random_state=SEED).fit_resample(X_tr, y_tr)

        m = make_tabnet()
        m.fit(X_train=X_tr.astype(np.float32), y_train=y_tr.astype(int),
              max_epochs=100, patience=0, batch_size=32, virtual_batch_size=16,
              num_workers=0, drop_last=False)
        fold_aucs.append(roc_auc_score(y_va, m.predict_proba(X_va.astype(np.float32))[:, 1]))

    tabnet_cv.append(np.mean(fold_aucs))
    print(f'  seed {s}: {tabnet_cv[-1]:.3f}   ({time.time()-t0:.0f}s)')

print(f'\nTabNet CV AUC: {np.mean(tabnet_cv):.3f} ± {np.std(tabnet_cv):.3f}')
print('Expected (manuscript Table 3): 0.688 ± 0.012')

  seed 101: 0.683   (357s)
  seed 202: 0.682   (725s)
  seed 303: 0.705   (1094s)


## 7. Sampling uncertainty of single-split AUC (manuscript Section 3.3, Eq. 16)

Hanley–McNeil standard error of a nonparametric AUC. With n₊ = 7 and n₋ = 20, the
single-split estimates carry standard errors of roughly 0.07–0.09 — comparable to
the differences between the models in Table 3.

In [ ]:
def hanley_mcneil_se(A, n_pos=7, n_neg=20):
    """Manuscript Eq. (16)."""
    Q1, Q2 = A / (2 - A), 2 * A**2 / (1 + A)
    var = (A*(1-A) + (n_pos-1)*(Q1 - A**2) + (n_neg-1)*(Q2 - A**2)) / (n_pos*n_neg)
    return np.sqrt(var)

print(f'{"Configuration":30s} {"AUC":>6s} {"SE":>7s}   95% CI')
for label, A_ in [('CatBoost baseline', 0.907), ('TabNet (configuration A)', auc_A)]:
    se = hanley_mcneil_se(A_)
    print(f'{label:30s} {A_:6.3f} {se:7.3f}   '
          f'[{A_ - 1.96*se:.3f}, {min(1.0, A_ + 1.96*se):.3f}]')

d    = auc_A - 0.907
se_d = np.sqrt(hanley_mcneil_se(auc_A)**2 + hanley_mcneil_se(0.907)**2)
print(f'\nDifference {d:+.3f}, SE {se_d:.3f}, z = {d/se_d:.2f}')
print('No pair of models in Table 3 is separable by this protocol.')